In [29]:
import pandas as pd
import re

def clean_text(text):
    text = re.sub(r'\b\w+:\s*[\w\-.]+\b', '', text)  # prosta regexowa próba usunięcia "Key:Value"
    text = re.sub(r'[;]', ',', text)  # średniki na przecinki
    text = re.sub(r'\s+', ' ', text)  # usuń nadmiarowe spacje
    return text.strip()

bsdd_df = pd.read_csv("../data/contrastive_dataset/file02.csv")
ifc_files = ["../data/contrastive_dataset/file01.csv", ]
dfs = [pd.read_csv(path) for path in ifc_files]  
ifc_df = pd.concat(dfs, ignore_index=True)

# Czyść teksty
bsdd_texts = bsdd_df["Text"].dropna().apply(clean_text).tolist()
ifc_texts = ifc_df["Text"].dropna().apply(clean_text).tolist()

# Dodaj nazwy (jeśli chcesz)
bsdd_names = bsdd_df["Name"].dropna().tolist()
ifc_names = ifc_df["Name"].dropna().tolist()

corpus = bsdd_texts + bsdd_names + ifc_texts + ifc_names
corpus = [c for c in corpus if len(c) > 10]  

with open("../data/contrastive_dataset/corpus_v05.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

print(f"Zapisano {len(corpus)} lini do corpus_v05.txt")

Zapisano 5456 lini do corpus_v05.txt


In [32]:
import pandas as pd
import random

# Ścieżki plików
ifc_files = [
    "../data/contrastive_dataset/file01.csv",
    "../data/contrastive_dataset/file03.csv",
    "../data/contrastive_dataset/file04.csv",
    "../data/contrastive_dataset/file05.csv",
    "../data/contrastive_dataset/file06.csv"
]
bsdd_file = "../data/contrastive_dataset/file02.csv"

# Wczytanie IFC
ifc_dfs = [pd.read_csv(f) for f in ifc_files]
ifc = pd.concat(ifc_dfs, ignore_index=True)

# Wczytanie bSDD
bsdd = pd.read_csv(bsdd_file)

# Kolumny tekstowe do łączenia
ifc_text_cols = ["Name", "Text"]
bsdd_text_cols = ["Code", "Name", "Text"]

# Tworzenie czystego opisu
ifc["full_text"] = ifc[ifc_text_cols].fillna("").agg(" ".join, axis=1)
bsdd["full_text"] = bsdd[bsdd_text_cols].fillna("").agg(" ".join, axis=1)

# Usuwanie duplikatów i pustych
ifc = ifc[ifc["full_text"].str.strip() != ""].drop_duplicates(subset="full_text")
bsdd = bsdd[bsdd["full_text"].str.strip() != ""].drop_duplicates(subset="full_text")

# Balansowanie zbioru
min_size = min(len(ifc), len(bsdd))
ifc_balanced = ifc.sample(min_size, random_state=42)
bsdd_balanced = bsdd.sample(min_size, random_state=42)

# Połączenie
final_corpus = list(ifc_balanced["full_text"]) + list(bsdd_balanced["full_text"])
random.shuffle(final_corpus)

# Zapis do pliku
with open("../data/contrastive_dataset/corpus_balanced.txt", "w", encoding="utf-8") as f:
    for line in final_corpus:
        f.write(line.strip() + "\n")

print(f"Zapisano {len(final_corpus)} linii do corpus_balanced.txt")

Zapisano 4046 linii do corpus_balanced.txt


In [35]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import math

model_name = "paraphrase-multilingual-MiniLM-L12-v2"
batch_size = 32
epochs = 2
learning_rate = 2e-5

# Wczytanie danych
with open("contrastive_pairs.csv", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

train_samples = [InputExample(texts=[line, line]) for line in lines]
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=batch_size)

model = SentenceTransformer(model_name)

train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(0.05 * len(train_dataloader) * epochs)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': learning_rate},
    show_progress_bar=True,
    output_path="../models/contrastive_paired"
)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2
                                                                                                                       

Step,Training Loss
500,0.121800


INFO:sentence_transformers.SentenceTransformer:Save model to ../models/contrastive_paired
